In [22]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


In [2]:
df = pd.read_csv("../Data/student-mental-health.csv")
df.head(6)

,Student_ID,Age,Gender,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Physical_Activity,Financial_Stress,Family_Support,Attendance_Rate,Part_Time_Job,Diet_Quality,Previous_Mental_Health_History,Mental_Health_Risk
0,STU0001,22,Male,3,6.6,17.6,2.91,6.9,7.5,Moderate,8.5,4.0,78.0,No,Poor,No,High
1,STU0002,21,Male,4,5.6,22.4,2.92,7.6,11.5,Low,4.5,5.3,85.4,Yes,Average,No,Medium
2,STU0003,22,Male,5,7.1,28.1,2.68,6.5,8.9,Low,4.8,6.1,79.5,No,Average,No,Medium
3,STU0004,24,Male,1,4.5,14.8,2.23,5.5,7.6,High,3.7,8.0,73.9,No,Average,Yes,High
4,STU0005,20,Female,1,6.7,17.2,2.69,4.3,7.0,Moderate,8.0,7.4,99.8,Yes,NaN,No,Medium
5,STU0006,20,Female,4,6.8,18.6,NaN,8.5,11.8,High,7.0,3.9,NaN,Yes,Average,No,Medium


In [5]:
X = df.drop(columns=["Student_ID","Mental_Health_Risk"],errors='ignore')
y = df["Mental_Health_Risk"]

In [9]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [11]:
numeric_cols = [
    'Sleep_Hours',
    'GPA',
    'Social_Media_Hours',
    'Financial_Stress',
    'Family_Support',
    'Attendance_Rate'
]

categorical_cols = [
    'Diet_Quality'
]

In [12]:
mediane_imputer = SimpleImputer(strategy='median')

X_train[numeric_cols]= mediane_imputer.fit_transform(X_train[numeric_cols])
X_test[numeric_cols]=mediane_imputer.transform(X_test[numeric_cols])

mode_imputer = SimpleImputer(strategy='most_frequent')
X_train[categorical_cols]=mode_imputer.fit_transform(X_train[categorical_cols])
X_test[categorical_cols]=mode_imputer.transform(X_test[categorical_cols])

In [23]:
numeric_cols = df.select_dtypes(include="number").columns.to_list()
nominal_cols = ['Gender', 'Part_Time_Job', 'Previous_Mental_Health_History']
ordinal_cols = ['Diet_Quality', 'Physical_Activity']

categories_ordinales = [
    ["Poor","Average","Good"],
    ["Low","Moderate","High"]   
]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),      
    ('scaler', StandardScaler())                       
])

nominal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), 
    ('onehot', OneHotEncoder(handle_unknown="ignore", sparse_output=False)) 
])

ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), 
    ('ordinal', OrdinalEncoder(categories=categories_ordinales)) 
])


preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('nom', nominal_transformer, nominal_cols),
        ('ord', ordinal_transformer, ordinal_cols)
    ]
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor)
])

X_train_processed = pipeline.fit_transform(X_train)
X_test_processed = pipeline.transform(X_test)

In [24]:
import pickle

with open('donnees_et_preprocesseur.pkl', 'wb') as f:
    pickle.dump((X_train_processed, X_test_processed, y_train, y_test, preprocessor), f)

print("Donnees sauvegardes")

Donnees sauvegardes
